In [1]:
import sys
from pathlib import Path

# !pip install -r requirements.txt

import pandas as pd
import networkx as nx
import re
import yaml
from itertools import chain
from pathlib import Path
from operator import itemgetter
from collections import defaultdict
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# !pip install mygene
import mygene
mg = mygene.MyGeneInfo()




In [2]:
os.getcwd()


'/home/jjoy/projects/DrugMechDB/data_analysis'

In [3]:
os.getcwd()

os.chdir("/home/jjoy/projects/DrugMechDB/")

os.getcwd()



'/home/jjoy/projects/DrugMechDB'

In [4]:
import sys
print(sys.path)

sys.path.append('/home/jjoy/projects/DrugMechDB')

['/home/jjoy/miniforge3/envs/dmdb/lib/python313.zip', '/home/jjoy/miniforge3/envs/dmdb/lib/python3.13', '/home/jjoy/miniforge3/envs/dmdb/lib/python3.13/lib-dynload', '', '/home/jjoy/miniforge3/envs/dmdb/lib/python3.13/site-packages']


In [5]:
# 
from data_tools.data_tools.files._retrieval import download
from data_tools.data_tools.files._analysis import *
from data_tools.data_tools.plotting import count_plot_h,venn2_pretty, venn3_pretty
#import libraries 

from pySankey.sankey import sankey
from tqdm import tqdm

from wordcloud import WordCloud, STOPWORDS

import warnings
warnings.filterwarnings("ignore")

In [6]:
# Make the output folders
this_name = '1_basic_dmdb_analysis'
out_dir = Path('../2_pipeline').joinpath(this_name, 'out').resolve()
out_dir.mkdir(parents=True, exist_ok=True)

data_dir = Path('../0_data/external').resolve()
data_dir.mkdir(parents=True, exist_ok=True)

In [7]:
DMDB_URL = 'https://raw.githubusercontent.com/SuLab/DrugMechDB/main/indication_paths.yaml'
download(DMDB_URL, data_dir.joinpath('indication_paths.yaml'), redownload=False)

File indication_paths.yaml exits. Skipping...


In [8]:
with open(data_dir.joinpath('indication_paths.yaml'), 'r') as fh:
        ind = yaml.safe_load(fh)

In [9]:
all_metapath_nodes = get_metapath_node(ind)
all_metapath_edges = get_metapath_edges(ind)

In [10]:
basic_stats = defaultdict(list)
all_metaedges = []
all_parings = []
all_targets = []
unique_metaedges = []
first_edge_type = []
all_nodes = []

id_to_name = {}
id_to_label = {}

for i, p in enumerate(ind):
    _id = (p["graph"]["_id"])
    drug_id, dis_id = path_to_tup(p)
    paths = get_all_paths(p)
    G = path_to_G(p)
    
    G = add_metaedges(G)
    G = add_meanode_pairs(G)
    
    basic_stats['idx'].append(i) #index
    basic_stats['id'].append(p['graph']['_id']) #DrugMechDB id
    basic_stats['drug'].append(drug_id) #Drug id
    basic_stats['disease'].append(dis_id)#Disease id
    basic_stats['nodes'].append((G.nodes)) #nodes in metapath
    basic_stats['n_nodes'].append(len(G.nodes)) # number of nodes in metapath
    basic_stats['n_edges'].append(len(G.edges)) #number of edges in metapath
    basic_stats['n_paths'].append(len(all_metapath_nodes[_id])) #number of paths
    basic_stats['metapath'].append(all_metapath_nodes[_id])
    basic_stats['metapath_with_edges'].append(all_metapath_edges[_id])

    
    this_metaedges = [G.edges[e]['metaedge'] for e in G.edges]
    
    all_metaedges += this_metaedges
    unique_metaedges += list(set(this_metaedges))
    
    all_parings += [G.edges[e]['mn_pair'] for e in G.edges]
    all_targets += get_targets(G)
    first_edge_type += get_target_metaedges(G)
    all_nodes += list(G.nodes)
    
    id_to_label = {**id_to_label, **get_id_to_type(G)}
    id_to_name = {**id_to_name, **get_id_to_name(G)}
    
basic_stats = pd.DataFrame(basic_stats)

In [11]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [12]:
basic_stats.head()

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease]
2,2,DB00316_MESH_D010146_1,DB:DB00316,MESH:D010146,"(MESH:D000082, UniProt:P23219, UniProt:P35354, UniProt:Q15185, GO:0001516, MESH:D011453, MESH:D010146)",7,8,1,[Drug - Protein - BiologicalProcess - ChemicalSubstance - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - increases abundance of - ChemicalSubstance - positively correlated with - Disease]
3,3,DB00316_MESH_D005334_1,DB:DB00316,MESH:D005334,"(MESH:D000082, reactome:R-HSA-2162123, UBERON:0000955, GO:0001659, MESH:D005334)",5,4,1,[Drug - Pathway - GrossAnatomicalStructure - BiologicalProcess - Disease],[Drug - negatively regulates - Pathway - occurs in - GrossAnatomicalStructure - location of - BiologicalProcess - negatively correlated with - Disease]
4,4,DB00945_MESH_D010146_1,DB:DB00945,MESH:D010146,"(MESH:D001241, UniProt:P23219, UniProt:P35354, GO:0001516, MESH:D011453, GO:0006954, MESH:D010146)",7,7,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease]


In [13]:
basic_stats["metapath_with_edges"].apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique()

1331

In [14]:
basic_stats["metapath_with_edges"].apply(lambda x: tuple(x) if isinstance(x, list) else x).unique()

array([('Drug - decreases activity of - Protein - causes - Disease',),
       ('Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease',),
       ('Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - increases abundance of - ChemicalSubstance - positively correlated with - Disease',),
       ...,
       ('Drug - positively regulates - Protein - increases abundance of - GeneFamily - positively correlated with - BiologicalProcess - positively correlated with - PhenotypicFeature - manifestation of - Disease',),
       ('Drug - positively regulates - Protein - increases abundance of - GeneFamily - positively correlated with - BiologicalProcess - caused by - OrganismTaxon - causes - Disease',),
       ('Drug - positively regulates - Protein - negatively correlated with - BiologicalProcess - positively correlated with - Disease',)],
      dtype=object)

In [15]:
basic_stats["metapath_with_edges"].apply(
    lambda x: isinstance(x, list) and len(x) > 1 and x[0].startswith("Drug") and x[-1].startswith("Disease")
).sum()

np.int64(0)

In [16]:
basic_stats = pd.DataFrame(basic_stats)

In [17]:
basic_stats.shape

(4846, 10)

In [18]:
def contains_target_string(metapath_with_edges_list, target_string):
    return target_string in metapath_with_edges_list

target_string = "Drug - decreases activity of - Protein - causes - Disease"

filtered_df = basic_stats[basic_stats['metapath_with_edges'].apply(lambda x: contains_target_string(x, target_string))]


In [19]:
filtered_df.shape

(3, 10)

In [20]:
filtered_df

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]
241,241,DB04865_MESH_D015464_1,DB:DB04865,MESH:D015464,"(MESH:C001652, UniProt:P39023, GO:0006412, UniProt:A9UF02, MESH:D015464)",5,4,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]
435,435,DB06616_MESH_D001752_1,DB:DB06616,MESH:D001752,"(MESH:C471992, UniProt:A9UF02, MESH:D001752)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]


In [21]:
basic_stats[(basic_stats["n_nodes"]==3) & (basic_stats["n_edges"]==2) & (basic_stats["n_paths"]==1)].head(5)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]
23,23,DB00326_MESH_D006996_1,DB:DB00326,MESH:D006996,"(MESH:C007852, MESH:D002118, MESH:D006996)",3,2,1,[Drug - ChemicalSubstance - Disease],[Drug - increases abundance of - ChemicalSubstance - prevents - Disease]
43,43,DB00650_MESH_D000749_1,DB:DB00650,MESH:D000749,"(MESH:D002955, MESH:D005492, MESH:D000749)",3,2,1,[Drug - ChemicalSubstance - Disease],[Drug - increases abundance of - ChemicalSubstance - prevents - Disease]
50,50,MESH_C106301_MESH_D003233_1,None,MESH:D003233,"(MESH:C106301, GO:0002553, MESH:D003233)",3,2,1,[Drug - BiologicalProcess - Disease],[Drug - negatively regulates - BiologicalProcess - correlated with - Disease]
58,58,DB06786_MESH_D012871_1,DB:DB06786,MESH:D012871,"(MESH:D006206, GO:0006954, MESH:D012871)",3,2,1,[Drug - BiologicalProcess - Disease],[Drug - negatively regulates - BiologicalProcess - causes - Disease]


In [22]:
basic_stats["n_paths"].value_counts()

n_paths
1     4258
2      440
4       48
3       41
0       29
5       15
6       13
21       1
10       1
Name: count, dtype: int64

In [23]:
type(basic_stats)

pandas.core.frame.DataFrame

In [24]:
print(basic_stats[["drug", "disease", "nodes"]].isna().sum())  # Check for NaNs
print(basic_stats[["drug", "disease", "nodes"]].head())  

drug       1
disease    0
nodes      0
dtype: int64
         drug       disease  \
0  DB:DB00619  MESH:D015464   
1  DB:DB00619  MESH:D034721   
2  DB:DB00316  MESH:D010146   
3  DB:DB00316  MESH:D005334   
4  DB:DB00945  MESH:D010146   

                                                                                                    nodes  
0                                                         (MESH:D000068877, UniProt:P00519, MESH:D015464)  
1                             (MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)  
2  (MESH:D000082, UniProt:P23219, UniProt:P35354, UniProt:Q15185, GO:0001516, MESH:D011453, MESH:D010146)  
3                        (MESH:D000082, reactome:R-HSA-2162123, UBERON:0000955, GO:0001659, MESH:D005334)  
4      (MESH:D001241, UniProt:P23219, UniProt:P35354, GO:0001516, MESH:D011453, GO:0006954, MESH:D010146)  


In [25]:
basic_stats[basic_stats["drug"].isna()]


,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
50,50,MESH_C106301_MESH_D003233_1,None,MESH:D003233,"(MESH:C106301, GO:0002553, MESH:D003233)",3,2,1,[Drug - BiologicalProcess - Disease],[Drug - negatively regulates - BiologicalProcess - correlated with - Disease]


In [26]:
basic_stats["drug"] = basic_stats["drug"].fillna("Unknown")

In [27]:
basic_stats[basic_stats["drug"].isna()]

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges


In [28]:
grouped = basic_stats.groupby(["drug", "disease"])
grouped_agg = grouped.size().reset_index(name="count")  # Example aggregation
print(grouped_agg.shape)

(4648, 3)


In [29]:
grouped_agg[grouped_agg["count"]>1].head(3)

,drug,disease,count
13,DB:DB00008,MESH:D019698,2
36,DB:DB00038,MESH:D013921,2
153,DB:DB00169,MESH:D014808,2


In [30]:
basic_stats["n_paths"].value_counts()

n_paths
1     4258
2      440
4       48
3       41
0       29
5       15
6       13
21       1
10       1
Name: count, dtype: int64

In [31]:
import ast

In [32]:
basic_stats[basic_stats["id"]=="DB00674_MESH_D000544_1"]

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
2805,2805,DB00674_MESH_D000544_1,DB:DB00674,MESH:D000544,"(MESH:D005702, UniProt:P22303, GO:0006581, CHEBI:15355, HP:0100543, MESH:D000544)",6,5,1,[Drug - Protein - BiologicalProcess - ChemicalSubstance - PhenotypicFeature - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - decreases abundance of - ChemicalSubstance - negatively correlated with - PhenotypicFeature - manifestation of - Disease]


# Benchmark Creation

### Drug- Biological Process dataset creation

Which biological process is altered by amoxicillin in the treatment of Streptococcal tonsillitis?

Which Drug can be used in the treatment of Bacterial septicemia by targeting bacterial Nucleic Acid synthesis?

In [33]:
def count_go_occurrences(nodes_set):
    # Count how many times 'GO' appears in the nodes and filter for them
    return sum(1 for node in nodes_set if 'GO:' in node)

# Filter DataFrame
go_bp = basic_stats[basic_stats['nodes'].apply(count_go_occurrences)==1]

go_bp.shape

(1840, 10)

In [34]:
go_bp.iloc[0:1]

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease]


In [35]:
def extract_go_ids(nodes):
    return ', '.join([node for node in nodes if node.startswith('GO:')])

go_bp['bp'] = go_bp['nodes'].apply(extract_go_ids)

In [36]:
go_bp['Drug_MeshID'] = go_bp['nodes'].apply(lambda x: list(x)[0])

In [37]:
go_bp.iloc[0:1]

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,bp,Drug_MeshID
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease],GO:0008283,MESH:D000068877


In [38]:
def get_names(row, id_to_name):
    drug_name = id_to_name.get(row['Drug_MeshID'], 'Unknown')
    disease_name = id_to_name.get(row['disease'], 'Unknown')
    
    # Handle multiple protein IDs
    bp_ids = row['bp'].split(', ') if isinstance(row['bp'], str) else [row['bp']]
    bp_names = [id_to_name.get(bp_id, 'Unknown') for bp_id in bp_ids]

    return pd.Series([drug_name, disease_name, bp_names])


In [39]:
def map_node_names_from_nodeview(node_view, id_to_name):
    if not hasattr(node_view, '__iter__'):
        return ['Unknown']
    
    return [id_to_name.get(str(node_id).strip(), 'Unknown') for node_id in node_view]


In [40]:
go_bp['node_names'] = go_bp['nodes'].apply(lambda x: map_node_names_from_nodeview(x, id_to_name))


In [41]:
go_bp.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,bp,Drug_MeshID,node_names
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease],GO:0008283,MESH:D000068877,"[imatinib, Mast/stem cell growth factor receptor Kit, Platelet-derived growth factor receptor alpha, cell population proliferation, Systemic mast cell disease]"


In [42]:
go_bp[['drug_name', 'disease_name', 'bp_name']] = go_bp.apply(get_names, axis=1, id_to_name=id_to_name)

In [43]:
go_bp.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,bp,Drug_MeshID,node_names,drug_name,disease_name,bp_name
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease],GO:0008283,MESH:D000068877,"[imatinib, Mast/stem cell growth factor receptor Kit, Platelet-derived growth factor receptor alpha, cell population proliferation, Systemic mast cell disease]",imatinib,Systemic mast cell disease,[cell population proliferation]


In [44]:
# convert bp_name into str
go_bp['bp_name'] = go_bp['bp_name'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

In [45]:
go_bp.shape

(1840, 16)

In [46]:

go_bp['question'] = "Which Drug can be used in the treatment of " + go_bp['disease_name'] + " by targeting biological process: " + go_bp['bp_name'] + "?"

In [47]:
go_bp.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,bp,Drug_MeshID,node_names,drug_name,disease_name,bp_name,question
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease],GO:0008283,MESH:D000068877,"[imatinib, Mast/stem cell growth factor receptor Kit, Platelet-derived growth factor receptor alpha, cell population proliferation, Systemic mast cell disease]",imatinib,Systemic mast cell disease,cell population proliferation,Which Drug can be used in the treatment of Systemic mast cell disease by targeting biological process: cell population proliferation?


In [48]:
go_bp["n_nodes"].value_counts()

n_nodes
5     655
6     496
7     231
4     171
8     136
9      47
3      41
10     36
11     16
13      5
12      4
14      2
Name: count, dtype: int64

#### for the drug-centric benchmark, for now will take nodes greaster than 2 and less than 6 to capture mechanistic relationships without them getting too complex with increasing the number of intermediate nodes

In [49]:
go_bp["n_paths"].value_counts()

n_paths
1    1704
2     116
5       8
0       6
3       4
4       2
Name: count, dtype: int64

In [50]:
go_bp["Drug_MeshID"] = go_bp["Drug_MeshID"].str.replace(
    r"^DB:", "DRUGBANK:", regex=True
)

In [51]:
go_bp["drug"] = go_bp["drug"].str.replace(
    r"^DB:", "DRUGBANK:", regex=True
)

In [52]:
# filter go_bp with n_paths == 1 and n_nodes >2 and n_nodes < 6
go_bp_filtered = go_bp[(go_bp["n_paths"] == 1) & (go_bp["n_nodes"] > 2) & (go_bp["n_nodes"] < 6)]
go_bp_filtered.shape

(842, 17)

In [53]:
columns_order = ['idx', 'id', 'drug', 'Drug_MeshID', 'disease', 'bp','drug_name','disease_name','bp_name','nodes', 'n_nodes', 'n_edges', 'n_paths', 'metapath', 'metapath_with_edges', 'question']
go_bp_filtered = go_bp_filtered[columns_order]

In [54]:
go_bp_filtered.head(1)

,idx,id,drug,Drug_MeshID,disease,bp,drug_name,disease_name,bp_name,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,question
1,1,DB00619_MESH_D034721_1,DRUGBANK:DB00619,MESH:D000068877,MESH:D034721,GO:0008283,imatinib,Systemic mast cell disease,cell population proliferation,"(MESH:D000068877, UniProt:P10721, UniProt:P16234, GO:0008283, MESH:D034721)",5,5,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - causes - Disease],Which Drug can be used in the treatment of Systemic mast cell disease by targeting biological process: cell population proliferation?


In [55]:
go_bp_filtered.shape

(842, 16)

In [56]:
# go_bp_filtered.to_csv("_data/DMDB_go_bp_filtered.csv", index=False)

### Metabolite dataset

In [147]:
def count_chebi_occurrences(nodes_set):
    # Count how many times 'CEHBI' appears in the nodes 
    return sum(1 for node in nodes_set if 'CHEBI:' in node)

# Filter DataFrame
chebi_metabolite = basic_stats[basic_stats['nodes'].apply(count_chebi_occurrences)==1]

chebi_metabolite.shape

(327, 10)

In [148]:
chebi_metabolite.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
56,56,DB00994_MESH_D007634_1,DB:DB00994,MESH:D007634,"(MESH:D009355, CHEBI:18111, GO:0006412, taxonomy:1280, MESH:D007634)",5,4,1,[Drug - ChemicalSubstance - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - ChemicalSubstance - participates in - BiologicalProcess - in taxon - OrganismTaxon - causes - Disease]


In [149]:
def extract_chebi_ids(nodes):
    return ', '.join([node for node in nodes if node.startswith('CHEBI:')])

chebi_metabolite['metabolite'] = chebi_metabolite['nodes'].apply(extract_chebi_ids)

In [150]:
chebi_metabolite.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,metabolite
56,56,DB00994_MESH_D007634_1,DB:DB00994,MESH:D007634,"(MESH:D009355, CHEBI:18111, GO:0006412, taxonomy:1280, MESH:D007634)",5,4,1,[Drug - ChemicalSubstance - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - ChemicalSubstance - participates in - BiologicalProcess - in taxon - OrganismTaxon - causes - Disease],CHEBI:18111


In [151]:
chebi_metabolite['Drug_MeshID'] = chebi_metabolite['nodes'].apply(lambda x: list(x)[0])

In [152]:
def get_names(row, id_to_name):
    drug_name = id_to_name.get(row['Drug_MeshID'], 'Unknown')
    disease_name = id_to_name.get(row['disease'], 'Unknown')
    
    # Handle multiple protein IDs
    metabolite_ids = row['metabolite'].split(', ') if isinstance(row['metabolite'], str) else [row['metabolite']]
    metabolite_names = [id_to_name.get(metabolite_id, 'Unknown') for metabolite_id in metabolite_ids]

    return pd.Series([drug_name, disease_name, metabolite_names])


In [153]:
id_to_name.get("CHEBI:18111")

'ribosomal RNA'

In [154]:
print(repr(chebi_metabolite['nodes'].iloc[0]))


NodeView(('MESH:D009355', 'CHEBI:18111', 'GO:0006412', 'taxonomy:1280', 'MESH:D007634'))


In [155]:
def map_node_names_from_nodeview(node_view, id_to_name):
    if not hasattr(node_view, '__iter__'):
        return ['Unknown']
    
    return [id_to_name.get(str(node_id).strip(), 'Unknown') for node_id in node_view]


In [156]:
chebi_metabolite['node_names'] = chebi_metabolite['nodes'].apply(lambda x: map_node_names_from_nodeview(x, id_to_name))


In [157]:
chebi_metabolite.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,metabolite,Drug_MeshID,node_names
56,56,DB00994_MESH_D007634_1,DB:DB00994,MESH:D007634,"(MESH:D009355, CHEBI:18111, GO:0006412, taxonomy:1280, MESH:D007634)",5,4,1,[Drug - ChemicalSubstance - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - ChemicalSubstance - participates in - BiologicalProcess - in taxon - OrganismTaxon - causes - Disease],CHEBI:18111,MESH:D009355,"[neomycin, ribosomal RNA, translation, Staphylococcus aureus, Keratitis]"


In [158]:
# Check if 'ChemicalSubstance' is present in each row
has_chemical = chebi_metabolite['metapath_with_edges'].apply(lambda x: 'ChemicalSubstance' in str(x))

# Count how many rows have it and how many don't
count_with = has_chemical.sum()
count_without = (~has_chemical).sum()

print(f"Rows with 'ChemicalSubstance': {count_with}")
print(f"Rows without 'ChemicalSubstance': {count_without}")


Rows with 'ChemicalSubstance': 303
Rows without 'ChemicalSubstance': 24


In [159]:
chebi_metabolite[~chebi_metabolite['metapath_with_edges'].apply(lambda x: 'ChemicalSubstance' in str(x))].head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,metabolite,Drug_MeshID,node_names
542,542,DB00248_MESH_D006966_1,DB:DB00248,MESH:D006966,"(MESH:D000077465, UniProt:P14416, GO:0007195, CHEBI:17489, GO:0051209, GO:1902722, MESH:D006966)",7,6,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - increases activity of - Protein - participates in - BiologicalProcess - causes - Disease],CHEBI:17489,MESH:D000077465,"[Cabergoline, D(2) dopamine receptor, Adenylate cyclase-inhibiting dopamine receptor signaling pathway, 3',5'-cyclic AMP, Release of sequestered calcium ion into cytosol, Positive regulation of prolactin secretion, Hyperprolactinemia]"


In [160]:
chebi_metabolite[['drug_name', 'disease_name', 'metabolite_name']] = chebi_metabolite.apply(get_names, axis=1, id_to_name=id_to_name)

In [161]:
chebi_metabolite.shape

(327, 16)

In [162]:
chebi_metabolite["n_nodes"].value_counts()

n_nodes
7     83
5     55
8     48
6     34
4     21
10    18
11    18
9     18
3      9
13     8
12     7
14     5
15     2
18     1
Name: count, dtype: int64

In [163]:
def extract_edges(row):
    try:
        data = row['metapath_with_edges']
        
        # If it's a list with one string, flatten it
        if isinstance(data, list) and len(data) == 1 and isinstance(data[0], str):
            data = data[0]

        # If it's a string, parse it
        if isinstance(data, str):
            data = data.strip('[]')  # remove brackets
            elements = [x.strip() for x in data.split(' - ')]
        elif isinstance(data, list):
            elements = data  # already parsed
        else:
            return "Invalid format"
        
        # Edges are at odd indices
        edges = [elements[i] for i in range(1, len(elements), 2)]
        return edges
    except Exception as e:
        return f"Error: {e}"



In [164]:

chebi_metabolite['edges'] = chebi_metabolite.apply(extract_edges, axis=1)

In [165]:
def filter_and_format_metabolite_df(
    df,
    include_chemical_substance=True,
    exact_n_paths=1,
    columns_order=None
):
    
    df = df.copy()

    # Normalize DrugBank IDs
    df["Drug_MeshID"] = df["Drug_MeshID"].str.replace(r"^DB:", "DRUGBANK:", regex=True)
    df["drug"] = df["drug"].str.replace(r"^DB:", "DRUGBANK:", regex=True)

    # Filter by node count
    # df = df[(df["n_nodes"] >= min_nodes) & (df["n_nodes"] <= max_nodes)]

    # Exclude subclass metapaths
    df = df[~df["metapath_with_edges"].astype(str).str.contains("subclass")]

    # Include/exclude ChemicalSubstance
    if include_chemical_substance:
        df = df[df["metapath_with_edges"].astype(str).str.contains("ChemicalSubstance")]
    else:
        df = df[~df["metapath_with_edges"].astype(str).str.contains("ChemicalSubstance")]

    # Filter by exact path count
    if exact_n_paths is not None:
        df = df[df["n_paths"] == exact_n_paths]

    # Reorder columns if specified
    if columns_order:
        df = df[[col for col in columns_order if col in df.columns]]

    return df

In [166]:
chebi_metabolite.shape

(327, 17)

In [167]:
result_df = filter_and_format_metabolite_df(chebi_metabolite)
result_df.shape

(201, 17)

In [168]:
result_df["n_nodes"].value_counts()

n_nodes
7     63
5     44
8     28
6     27
4     18
9      9
3      6
10     2
12     1
11     1
14     1
13     1
Name: count, dtype: int64

In [169]:
result_df.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,metabolite,Drug_MeshID,node_names,drug_name,disease_name,metabolite_name,edges
56,56,DB00994_MESH_D007634_1,DRUGBANK:DB00994,MESH:D007634,"(MESH:D009355, CHEBI:18111, GO:0006412, taxonomy:1280, MESH:D007634)",5,4,1,[Drug - ChemicalSubstance - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - ChemicalSubstance - participates in - BiologicalProcess - in taxon - OrganismTaxon - causes - Disease],CHEBI:18111,MESH:D009355,"[neomycin, ribosomal RNA, translation, Staphylococcus aureus, Keratitis]",neomycin,Keratitis,[ribosomal RNA],"[decreases activity of, participates in, in taxon, causes]"


In [170]:
result_df.shape

(201, 17)

In [176]:
result_df.to_csv("_data/DMDB_chebi_metabolite_filtered.csv", index=False)

In [79]:
#result_df.to_csv("_data/DMDB_chebi_metabolite_filtered_04_23_2025.csv", index=False)

### Gene-centric dataset

In [82]:
basic_stats.shape

(4846, 10)

In [84]:
basic_stats.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]


In [86]:
def count_uniprot_occurrences(nodes_set):
    # Count how many times 'UniProt' appears in the nodes (which is a set in this case)
    return sum(1 for node in nodes_set if 'UniProt' in node)

# Filter DataFrame
filtered_df = basic_stats[basic_stats['nodes'].apply(count_uniprot_occurrences) == 1]

In [ ]:
# unique metapth find in basic_stats[(basic_stats["uniprot_count"]==1) & (basic_stats["n_paths"]==1)]["metapath"]
#basic_stats[(basic_stats["uniprot_count"]==1) & (basic_stats["n_paths"]==1)]["metapath"].value_counts()

In [89]:
filtered_df.shape

(1802, 10)

In [90]:
# filtered_df.groupby(["drug", "disease"]).head(1)

In [91]:
filtered_df["n_paths"].value_counts()

n_paths
1     1523
2      200
4       27
0       18
3       15
5       11
6        7
10       1
Name: count, dtype: int64

In [92]:
filtered_df.head(1)

,idx,id,drug,disease,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease]


In [93]:
def extract_uniprot_ids(nodes):
    return ', '.join([node for node in nodes if node.startswith('UniProt:')])

filtered_df['protein'] = filtered_df['nodes'].apply(extract_uniprot_ids)

In [94]:
columns_order = ['idx', 'id', 'drug', 'disease', 'protein', 'nodes', 'n_nodes', 'n_edges', 'n_paths', 'metapath', 'metapath_with_edges']
filtered_df = filtered_df[columns_order]
filtered_df['Drug_MeshID'] = filtered_df['nodes'].apply(lambda x: list(x)[0])
filtered_df.head(1)

,idx,id,drug,disease,protein,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,Drug_MeshID
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,UniProt:P00519,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease],MESH:D000068877


In [95]:
def get_names(row, id_to_name):
    drug_name = id_to_name.get(row['Drug_MeshID'], 'Unknown')
    disease_name = id_to_name.get(row['disease'], 'Unknown')
    protein_name = id_to_name.get(row['protein'], 'Unknown')
    
    return pd.Series([drug_name, disease_name, protein_name])

filtered_df[['drug_name', 'disease_name', 'protein_name']] = filtered_df.apply(get_names, axis=1, id_to_name=id_to_name)

In [96]:
filtered_df.head(1)

,idx,id,drug,disease,protein,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,Drug_MeshID,drug_name,disease_name,protein_name
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,UniProt:P00519,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease],MESH:D000068877,imatinib,Chronic myeloid leukemia,Tyrosine-protein kinase ABL1


In [97]:
def convert_protein_to_gene_symbol(uniprot_id):
    # Remove 'UniProt:' prefix
    uniprot_id = uniprot_id.replace('UniProt:', '')
    
    # Query mygene to get gene symbol
    gene_info = mg.query(uniprot_id, scopes='uniprot', fields='symbol', species='human')
    
    if gene_info and 'hits' in gene_info and gene_info['hits']:
        return gene_info['hits'][0].get('symbol', 'N/A')
    return 'N/A'

filtered_df['protein_gene_symbol'] = filtered_df['protein'].apply(convert_protein_to_gene_symbol)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

...

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

In [98]:
filtered_df.head(5)

,idx,id,drug,disease,protein,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,Drug_MeshID,drug_name,disease_name,protein_name,protein_gene_symbol
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,UniProt:P00519,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease],MESH:D000068877,imatinib,Chronic myeloid leukemia,Tyrosine-protein kinase ABL1,ABL1
5,5,DB00945_MESH_D013927_1,DB:DB00945,MESH:D013927,UniProt:P23219,"(MESH:D001241, UniProt:P23219, reactome:R-HSA-2162123, MESH:D013928, GO:0007596, MESH:D013927)",6,6,2,"[Drug - Protein - Pathway - ChemicalSubstance - BiologicalProcess - Disease, Drug - Protein - BiologicalProcess - Disease]","[Drug - decreases activity of - Protein - participates in - Pathway - has output - ChemicalSubstance - participates in - BiologicalProcess - causes - Disease, Drug - decreases activity of - Protein - participates in - BiologicalProcess - causes - Disease]",MESH:D001241,Aspirin,Thrombosis,Prostaglandin G/H synthase 1,PTGS1
6,6,DB00788_MESH_D010146_1,DB:DB00788,MESH:D010146,UniProt:P35354,"(MESH:D009288, UniProt:P35354, MESH:D011453, GO:0006954, MESH:D010146)",5,4,1,[Drug - Protein - ChemicalSubstance - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - increases abundance of - ChemicalSubstance - participates in - BiologicalProcess - causes - Disease],MESH:D009288,naproxen,Pain,Prostaglandin G/H synthase 2,PTGS2
7,7,DB01059_MESH_D004405_1,DB:DB01059,MESH:D004405,UniProt:P43702,"(MESH:D009643, UniProt:P43702, GO:0003746, GO:0006260, GO:0006412, taxonomy:622, MESH:D004405)",7,7,2,"[Drug - Protein - BiologicalProcess - OrganismTaxon - Disease, Drug - MolecularActivity - BiologicalProcess - OrganismTaxon - Disease]","[Drug - decreases activity of - Protein - participates in - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease, Drug - negatively regulates - MolecularActivity - precedes - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease]",MESH:D009643,norfloxacin,"Dysentery, Bacillary",DNA topoisomerase 4 subunit A (Haemophilus influenzae),N/A
14,14,DB00479_MESH_D016920_1,DB:DB00479,MESH:D016920,UniProt:P0A7S3,"(MESH:D000583, UniProt:P0A7S3, GO:0006412, taxonomy:2, MESH:D016920)",5,4,1,[Drug - Protein - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - Protein - participates in - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease],MESH:D000583,amikacin,"Meningitis, Bacterial",30S ribosomal protein S12,N/A


In [99]:
filtered_df.shape

(1802, 16)

In [100]:
filtered_df = filtered_df[filtered_df["n_paths"]==1]

In [101]:
filtered_df[filtered_df['protein_gene_symbol'] == 'N/A'].shape

(256, 16)

In [102]:
filtered_df[filtered_df['protein_gene_symbol'] == 'N/A'].head(5)

,idx,id,drug,disease,protein,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,Drug_MeshID,drug_name,disease_name,protein_name,protein_gene_symbol
14,14,DB00479_MESH_D016920_1,DB:DB00479,MESH:D016920,UniProt:P0A7S3,"(MESH:D000583, UniProt:P0A7S3, GO:0006412, taxonomy:2, MESH:D016920)",5,4,1,[Drug - Protein - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - Protein - participates in - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease],MESH:D000583,amikacin,"Meningitis, Bacterial",30S ribosomal protein S12,N/A
18,18,DB01263_MESH_D009181_1,DB:DB01263,MESH:D009181,UniProt:P10613,"(MESH:C101425, UniProt:P10613, MESH:D004875, GO:0009277, MESH:D009181)",5,4,1,[Drug - Protein - ChemicalSubstance - CellularComponent - Disease],[Drug - decreases activity of - Protein - produces - ChemicalSubstance - located in - CellularComponent - correlated with - Disease],MESH:C101425,Posaconazole,Fungal Infection,Lanosterol 14-alpha-demethylase,N/A
27,27,DB01048_MESH_D015658_1,DB:DB01048,MESH:D015658,UniProt:P04585,"(MESH:C106538, MESH:C066928, UniProt:P04585, GO:0039693, MESH:D015658)",5,4,1,[Drug - ChemicalSubstance - Protein - BiologicalProcess - Disease],[Drug - increases abundance of - ChemicalSubstance - decreases activity of - Protein - participates in - BiologicalProcess - causes - Disease],MESH:C106538,abacavir,Human immunodeficiency virus infection,Gag-Pol polyprotein,N/A
32,32,DB01212_MESH_D006069_1,DB:DB01212,MESH:D006069,UniProt:P0A3M6,"(MESH:D002443, UniProt:P0A3M6, GO:0009252, taxonomy:485, MESH:D006069)",5,4,1,[Drug - Protein - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - Protein - positively regulates - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease],MESH:D002443,ceftriaxone,Gonorrhea,Penicillin-binding protein 2B,N/A
33,33,DB02703_MESH_D000881_1,DB:DB02703,MESH:D000881,UniProt:Q81VT3,"(MESH:D005672, UniProt:Q81VT3, GO:0006412, taxonomy:1392, MESH:D000881)",5,4,1,[Drug - Protein - BiologicalProcess - OrganismTaxon - Disease],[Drug - decreases activity of - Protein - participates in - BiologicalProcess - occurs in - OrganismTaxon - causes - Disease],MESH:D005672,fusidic acid,Anthrax,Elongation factor G,N/A


In [103]:
filtered_df.columns

Index(['idx', 'id', 'drug', 'disease', 'protein', 'nodes', 'n_nodes',
       'n_edges', 'n_paths', 'metapath', 'metapath_with_edges', 'Drug_MeshID',
       'drug_name', 'disease_name', 'protein_name', 'protein_gene_symbol'],
      dtype='object')

In [104]:
columns_order = ['idx', 'id', 'drug', 'Drug_MeshID', 'disease', 'protein',	'drug_name','disease_name',	'protein_name','protein_gene_symbol','nodes', 'n_nodes', 'n_edges', 'n_paths', 'metapath', 'metapath_with_edges']
filtered_df = filtered_df[columns_order]

In [105]:
filtered_df["question"] = "Which gene plays the most significant mechanistic role in how Drug " + filtered_df["drug_name"] + " treats or impacts the Disease " + filtered_df["disease_name"] + "?"
filtered_df.head(1)

,idx,id,drug,Drug_MeshID,disease,protein,drug_name,disease_name,protein_name,protein_gene_symbol,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,question
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D000068877,MESH:D015464,UniProt:P00519,imatinib,Chronic myeloid leukemia,Tyrosine-protein kinase ABL1,ABL1,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease],Which gene plays the most significant mechanistic role in how Drug imatinib treats or impacts the Disease Chronic myeloid leukemia?


In [106]:
filtered_df[['id','drug', 'disease',  'drug_name', 'disease_name','protein', 'protein_name', 'protein_gene_symbol', 'question']].head(3)

,id,drug,disease,drug_name,disease_name,protein,protein_name,protein_gene_symbol,question
0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,imatinib,Chronic myeloid leukemia,UniProt:P00519,Tyrosine-protein kinase ABL1,ABL1,Which gene plays the most significant mechanistic role in how Drug imatinib treats or impacts the Disease Chronic myeloid leukemia?
6,DB00788_MESH_D010146_1,DB:DB00788,MESH:D010146,naproxen,Pain,UniProt:P35354,Prostaglandin G/H synthase 2,PTGS2,Which gene plays the most significant mechanistic role in how Drug naproxen treats or impacts the Disease Pain?
14,DB00479_MESH_D016920_1,DB:DB00479,MESH:D016920,amikacin,"Meningitis, Bacterial",UniProt:P0A7S3,30S ribosomal protein S12,N/A,"Which gene plays the most significant mechanistic role in how Drug amikacin treats or impacts the Disease Meningitis, Bacterial?"


In [107]:
ids_to_select = ["DB08799_MESH_D012223_1", "DB00342_MESH_D017449_1", "DB01048_MESH_D015658_1"]
# print rows in filtered_df where id is in ids_to_select
filtered_df[filtered_df["id"].isin(ids_to_select)]

,idx,id,drug,Drug_MeshID,disease,protein,drug_name,disease_name,protein_name,protein_gene_symbol,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,question
19,19,DB08799_MESH_D012223_1,DB:DB08799,MESH:D000865,MESH:D012223,UniProt:P35367,Antazoline,"Rhinitis, Vasomotor",Histamine H1 receptor,HRH1,"(MESH:D000865, UniProt:P35367, GO:0034776, MESH:D012223)",4,3,1,[Drug - Protein - BiologicalProcess - Disease],[Drug - decreases activity of - Protein - participates in - BiologicalProcess - causes - Disease],"Which gene plays the most significant mechanistic role in how Drug Antazoline treats or impacts the Disease Rhinitis, Vasomotor?"
26,26,DB00342_MESH_D017449_1,DB:DB00342,MESH:D016593,MESH:D017449,UniProt:P35367,Terfenadine,allergic skin disorders,Histamine H1 receptor,HRH1,"(MESH:D016593, UniProt:P35367, HP:0000969, MESH:D017449)",4,3,1,[Drug - Protein - PhenotypicFeature - Disease],[Drug - decreases activity of - Protein - has phenotype - PhenotypicFeature - manifestation of - Disease],Which gene plays the most significant mechanistic role in how Drug Terfenadine treats or impacts the Disease allergic skin disorders?
27,27,DB01048_MESH_D015658_1,DB:DB01048,MESH:C106538,MESH:D015658,UniProt:P04585,abacavir,Human immunodeficiency virus infection,Gag-Pol polyprotein,N/A,"(MESH:C106538, MESH:C066928, UniProt:P04585, GO:0039693, MESH:D015658)",5,4,1,[Drug - ChemicalSubstance - Protein - BiologicalProcess - Disease],[Drug - increases abundance of - ChemicalSubstance - decreases activity of - Protein - participates in - BiologicalProcess - causes - Disease],Which gene plays the most significant mechanistic role in how Drug abacavir treats or impacts the Disease Human immunodeficiency virus infection?


In [108]:
filtered_df.shape

(1523, 17)

In [109]:
grouped_with_genes = (
    filtered_df.groupby(["disease", "Drug_MeshID"])
    .agg({
        "protein_gene_symbol": lambda x: ", ".join(set(x)),  # Concatenate unique symbols
        # Add any other columns to aggregate if needed
    })
    .reset_index()
)

# Add a count column to indicate the number of rows in each group
grouped_with_genes["count"] = filtered_df.groupby(["disease", "Drug_MeshID"]).size().values

In [110]:
filtered_df.head(1)

,idx,id,drug,Drug_MeshID,disease,protein,drug_name,disease_name,protein_name,protein_gene_symbol,nodes,n_nodes,n_edges,n_paths,metapath,metapath_with_edges,question
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D000068877,MESH:D015464,UniProt:P00519,imatinib,Chronic myeloid leukemia,Tyrosine-protein kinase ABL1,ABL1,"(MESH:D000068877, UniProt:P00519, MESH:D015464)",3,2,1,[Drug - Protein - Disease],[Drug - decreases activity of - Protein - causes - Disease],Which gene plays the most significant mechanistic role in how Drug imatinib treats or impacts the Disease Chronic myeloid leukemia?


In [111]:
grouped_with_genes = (
    filtered_df.groupby(["disease", "Drug_MeshID"])
    .agg({
        "protein_gene_symbol": lambda x: ", ".join(set(x)),  # Concatenate unique symbols
        # "id": lambda x: ", ".join(set(x)),  # Keeping all unique IDs
        "drug": lambda x: ", ".join(set(x)),  # Keeping all unique drugs
        "drug_name": lambda x: ", ".join(set(x)),  # Keeping all unique drug names
        "disease_name": lambda x: ", ".join(set(x)),  # Keeping all unique disease names
        "protein_name": lambda x: ", ".join(set(x)),  # Keeping all unique protein names
        # "nodes": lambda x: ", ".join(set(x)),  # Keeping all unique nodes
        # "n_nodes": "first",  # Taking the first value assuming it's consistent
        # "n_edges": "first",  # Taking the first value assuming it's consistent
        # "n_paths": "first",  # Taking the first value assuming it's consistent
        # "metapath": lambda x: ", ".join(set(x)),  # Keeping all unique metapaths
        # "metapath_with_edges": lambda x: ", ".join(set(x))  # Keeping all unique metapaths with edges
    })
    .reset_index()
)

# Add a count column to indicate the number of unique proteins in each group
grouped_with_genes["count"] = filtered_df.groupby(["disease", "Drug_MeshID"])["protein"].nunique().values


In [112]:
grouped_with_genes["count"].value_counts()

count
1    1422
2      21
3       7
6       1
4       1
5       1
Name: count, dtype: int64

In [113]:
# grouped_with_genes[grouped_with_genes["count"]>1]

In [114]:
filtered_df.shape

(1523, 17)

In [115]:
filtered_df[filtered_df["n_paths"]==1].shape

(1523, 17)

In [116]:
filtered_df = filtered_df[filtered_df["n_paths"]==1]

In [144]:
# save the filtered_df
# filtered_df.to_csv("_data/drugmechDB_mechanistic_genes_df.csv", index=False)

In [135]:
import mygene
mg = mygene.MyGeneInfo()

In [136]:
mg = mygene.MyGeneInfo()

# List of NCBI Gene IDs
gene_ids = ['80336', '246329', '31111', '8835', '79602', '23030', '54997', '5413', '2185', '5742', '2995', '10678']

gene_info = mg.querymany(gene_ids, scopes='entrezgene', fields='symbol', species='human')

gene_name_dict = {item['query']: item.get('symbol', 'N/A') for item in gene_info}
df = pd.DataFrame(list(gene_name_dict.items()), columns=['NCBI Gene ID', 'Gene Name'])

print(df)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found no hit:	['31111']


   NCBI Gene ID Gene Name
0         80336   PABPC1L
1        246329     STAC3
2         31111       N/A
3          8835     SOCS2
4         79602   ADIPOR2
5         23030     KDM4B
6         54997      TESC
7          5413   SEPTIN5
8          2185     PTK2B
9          5742     PTGS1
10         2995      GYPC
11        10678    B3GNT2


In [137]:
uniprot_ids = ['P00519', 'P35354', 'P0A7S3', 'Q12791', 'P10613']

# Query mygene to get gene symbols
gene_info = mg.querymany(uniprot_ids, scopes='uniprot', fields='symbol', species='human')

gene_name_dict = {item['query']: item.get('symbol', 'N/A') for item in gene_info}
df = pd.DataFrame(list(gene_name_dict.items()), columns=['UniProt ID', 'Gene Symbol'])

print(df)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
2 input query terms found no hit:	['P0A7S3', 'P10613']


  UniProt ID Gene Symbol
0     P00519        ABL1
1     P35354       PTGS2
2     P0A7S3         N/A
3     Q12791      KCNMA1
4     P10613         N/A
